In [16]:
# 👉 Si aún no tienes las librerías:
# !pip install pandas imbalanced-learn xgboost scikit-learn -q

import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from sklearn.metrics import classification_report

# 1️⃣ Carga de datos
df = pd.read_csv("todoscsvs_transformado.csv")

# 2️⃣ Crear columna 'target' a partir de las 16 columnas binarias
measures = list("ABCDEFGHIJKLMNÑO")
df["target"] = df[measures].idxmax(axis=1)
mask_nom = df[measures].sum(axis=1) == 0
df.loc[mask_nom, "target"] = "NoMeasure"

# 3️⃣ Separar features y etiqueta
X = df.drop(columns=measures + ["target"])
y = df["target"]

# 4️⃣ Imputación de missing
num_cols = X.select_dtypes(include=[np.number]).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
cat_cols = X.select_dtypes(exclude=[np.number]).columns
X[cat_cols] = X[cat_cols].fillna("Missing")

# 5️⃣ One-hot encoding de categóricas
X = pd.get_dummies(X, drop_first=True)

# 6️⃣ Transformar etiqueta a números
le = LabelEncoder()
y_num = le.fit_transform(y)

# 7️⃣ Train/test split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y_num,
    test_size=0.2,
    stratify=y_num,
    random_state=42
)

# 8️⃣ Escalado (mantener sparse-safe)
scaler = StandardScaler(with_mean=False)
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# 9️⃣ Oversample de minoritarias
ros = RandomOverSampler(random_state=42)
X_ov, y_ov = ros.fit_resample(X_train_s, y_train)

# 🔟 Undersample de todas las clases a 50% de la mayoritaria
counts = Counter(y_ov)
maj = max(counts.values())
target_n = int(maj * 0.5)
strategy = {cls: target_n for cls in counts}
rus = RandomUnderSampler(sampling_strategy=strategy, random_state=42)
X_res, y_res = rus.fit_resample(X_ov, y_ov)

print("Original:", Counter(y_train))
print("Remuestreado:", Counter(y_res))

# 1️⃣1️⃣ Preparar DMatrix para XGBoost CPU
dtrain = xgb.DMatrix(X_res, label=y_res)
dtest  = xgb.DMatrix(X_test_s, label=y_test)

# 1️⃣2️⃣ Parámetros y entrenamiento
params = {
    "objective":   "multi:softmax",
    "num_class":   len(le.classes_),
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "nthread":     8
}
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=200,
    evals=[(dtrain, "train"), (dtest, "eval")],
    early_stopping_rounds=20,
    verbose_eval=10
)

# 1️⃣3️⃣ Predicción y evaluación
y_pred_num = bst.predict(dtest).astype(int)
y_pred     = le.inverse_transform(y_pred_num)
y_test_lbl = le.inverse_transform(y_test)
print(classification_report(y_test_lbl, y_pred))

# 1️⃣4️⃣ Guardar modelo
bst.save_model("gbm_tipo_medida_cpu.json")


Original: Counter({np.int64(14): 129755, np.int64(8): 97910, np.int64(0): 23682, np.int64(2): 5867, np.int64(1): 2711, np.int64(5): 1342, np.int64(6): 613, np.int64(9): 546, np.int64(4): 198, np.int64(7): 170, np.int64(3): 151, np.int64(11): 83, np.int64(13): 77, np.int64(10): 35, np.int64(12): 19, np.int64(15): 5, np.int64(16): 3})
Remuestreado: Counter({np.int64(0): 64877, np.int64(1): 64877, np.int64(2): 64877, np.int64(3): 64877, np.int64(4): 64877, np.int64(5): 64877, np.int64(6): 64877, np.int64(7): 64877, np.int64(8): 64877, np.int64(9): 64877, np.int64(10): 64877, np.int64(11): 64877, np.int64(12): 64877, np.int64(13): 64877, np.int64(14): 64877, np.int64(15): 64877, np.int64(16): 64877})
[0]	train-mlogloss:1.81097	eval-mlogloss:1.74759
[10]	train-mlogloss:0.72096	eval-mlogloss:1.03980
[20]	train-mlogloss:0.50525	eval-mlogloss:0.87612
[30]	train-mlogloss:0.40436	eval-mlogloss:0.77843
[40]	train-mlogloss:0.34326	eval-mlogloss:0.72528
[50]	train-mlogloss:0.29908	eval-mlogloss:0.6

c:\Users\garci\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\garci\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\garci\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

              precision    recall  f1-score   support

           A       0.37      0.55      0.44      5921
           B       0.24      0.60      0.35       678
           C       0.39      0.77      0.52      1467
           D       0.36      0.34      0.35        38
           E       0.14      0.12      0.13        50
           F       0.13      0.42      0.20       335
           G       0.14      0.35      0.20       153
           H       0.40      0.23      0.29        43
           I       0.88      0.67      0.76     24477
           J       0.24      0.28      0.26       136
           K       0.00      0.00      0.00         9
           L       0.48      0.48      0.48        21
           M       0.00      0.00      0.00         4
           N       0.29      0.21      0.24        19
   NoMeasure       1.00      0.98      0.99     32439
           O       0.00      0.00      0.00         1
           Ñ       0.00      0.00      0.00         1

    accuracy              

In [18]:
# 👉 Instala dependencias si aún no lo has hecho:
# !pip install pandas scikit-learn imbalanced-learn xgboost -q

import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
import xgboost as xgb
from sklearn.metrics import classification_report

# 1️⃣ Carga de datos y creación de etiqueta original
df = pd.read_csv("todoscsvs_transformado.csv")
measures = list("ABCDEFGHIJKLMNÑO")
df["target"] = df[measures].idxmax(axis=1)
df.loc[df[measures].sum(axis=1) == 0, "target"] = "NoMeasure"

# 2️⃣ Agrupar clases raras (<100 muestras) en 'Other'
counts = df["target"].value_counts()
rare = counts[counts < 100].index.tolist()
df["target_grouped"] = df["target"].replace(rare, "Other")

# 3️⃣ Definir X e y
X = df.drop(columns=measures + ["target", "target_grouped"])
y = df["target_grouped"]

# 4️⃣ Imputación de missing values
num_cols = X.select_dtypes(include=[np.number]).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
cat_cols = X.select_dtypes(exclude=[np.number]).columns
X[cat_cols] = X[cat_cols].fillna("Missing")

# 5️⃣ One-hot encoding de categóricas
X = pd.get_dummies(X, drop_first=True)

# 6️⃣ Codificar etiqueta a valores numéricos
le = LabelEncoder()
y_enc = le.fit_transform(y)

# 7️⃣ Train/test split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)

# 8️⃣ Escalado (sparse-safe)
scaler = StandardScaler(with_mean=False)
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ────────────────
# 🅰️ HistGradientBoostingClassifier con sample weights
# ────────────────
# Calcula pesos inverso-frecuencia
train_counts = Counter(y_train)
total = sum(train_counts.values())
class_weight_hgb = {cls: total/c for cls, c in train_counts.items()}
sample_weight_hgb = np.array([class_weight_hgb[c] for c in y_train])

hgb = HistGradientBoostingClassifier(
    loss="log_loss",
    max_iter=200,
    random_state=42
)
hgb.fit(X_train_s, y_train, sample_weight=sample_weight_hgb)

y_pred_hgb_num = hgb.predict(X_test_s)
y_pred_hgb     = le.inverse_transform(y_pred_hgb_num)
y_test_lbl     = le.inverse_transform(y_test)

print("=== HistGradientBoostingClassifier con pesos ===")
print(classification_report(y_test_lbl, y_pred_hgb))

# ────────────────
# 🅱️ XGBoost multiclass con pesos de muestra
# ────────────────
# Reusar los mismos pesos de clase para XGBoost
sample_weight_xgb = np.array([class_weight_hgb[c] for c in y_train])

dtrain = xgb.DMatrix(X_train_s, label=y_train, weight=sample_weight_xgb)
dtest  = xgb.DMatrix(X_test_s,  label=y_test)

params = {
    "objective":   "multi:softmax",
    "num_class":   len(le.classes_),
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "nthread":     8
}
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=200,
    evals=[(dtrain, "train"), (dtest, "eval")],
    early_stopping_rounds=20,
    verbose_eval=10
)

y_pred_xgb_num = bst.predict(dtest).astype(int)
y_pred_xgb     = le.inverse_transform(y_pred_xgb_num)

print("=== XGBoost multiclass con pesos ===")
print(classification_report(y_test_lbl, y_pred_xgb))

# 〓 Guardar modelo XGBoost
bst.save_model("gbm_tipo_medida_weighted.json")


=== HistGradientBoostingClassifier con pesos ===
              precision    recall  f1-score   support

           A       0.36      0.43      0.39      5921
           B       0.14      0.70      0.23       678
           C       0.29      0.78      0.43      1467
           D       0.15      0.47      0.22        38
           E       0.03      0.40      0.06        50
           F       0.07      0.59      0.12       335
           G       0.07      0.57      0.12       153
           H       0.03      0.35      0.06        43
           I       0.89      0.43      0.58     24477
           J       0.03      0.35      0.06       136
           L       0.12      0.67      0.20        21
   NoMeasure       1.00      0.98      0.99     32439
       Other       0.03      0.44      0.06        34

    accuracy                           0.71     65792
   macro avg       0.25      0.55      0.27     65792
weighted avg       0.86      0.71      0.75     65792

[0]	train-mlogloss:1.85791	eva

In [20]:
# 👉 Instala dependencias si aún no las tienes:
!pip install pandas scikit-learn catboost -q

import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1️⃣ Carga y creación de la etiqueta original
df = pd.read_csv("todoscsvs_transformado.csv")
measures = list("ABCDEFGHIJKLMNÑO")
df["target"] = df[measures].idxmax(axis=1)
df.loc[df[measures].sum(axis=1) == 0, "target"] = "NoMeasure"

# 2️⃣ Agrupar clases raras (<100 muestras) en 'Other'
counts = df["target"].value_counts()
rare = counts[counts < 100].index.tolist()
df["target_grouped"] = df["target"].replace(rare, "Other")

# 3️⃣ Separar características (X) y etiqueta (y)
X = df.drop(columns=measures + ["target", "target_grouped"])
y = df["target_grouped"]

# 4️⃣ Imputación básica de valores faltantes
num_cols = X.select_dtypes(include=[np.number]).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
# Las columnas categóricas pueden quedar con NaN; CatBoost las maneja internamente

# 5️⃣ Identificar columnas categóricas para CatBoost
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

# 6️⃣ División train/test estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# 7️⃣ Calcular pesos inverso-frecuencia para cada clase
train_counts = y_train.value_counts()
total = len(y_train)
class_weights = {cls: total/count for cls, count in train_counts.items()}

# 8️⃣ Crear Pools de CatBoost (incluyendo pesos de muestra)
train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=cat_features,
    weight=y_train.map(class_weights)
)
eval_pool = Pool(
    data=X_test,
    label=y_test,
    cat_features=cat_features
)

# 9️⃣ Configurar y entrenar el modelo
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    loss_function="MultiClass",
    eval_metric="TotalF1",           # Macro-F1 para multiclass
    class_weights=class_weights,     # Ajuste interno de la pérdida
    random_seed=42,
    early_stopping_rounds=50,
    verbose=20
)
model.fit(train_pool, eval_set=eval_pool)

# 🔟 Predicción y evaluación
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))

# 1️⃣1️⃣ Guardar modelo
model.save_model("catboost_tipo_medida.cbm")



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


0:	learn: 0.3724653	test: 0.0802659	best: 0.0802659 (0)	total: 2.52s	remaining: 20m 56s
20:	learn: 0.5589417	test: 0.1056503	best: 0.1067796 (19)	total: 39s	remaining: 14m 49s
40:	learn: 0.6486931	test: 0.1423471	best: 0.1423471 (40)	total: 1m 15s	remaining: 13m 59s
60:	learn: 0.7106521	test: 0.1553328	best: 0.1553328 (60)	total: 1m 53s	remaining: 13m 37s
80:	learn: 0.7326418	test: 0.1716673	best: 0.1716673 (80)	total: 2m 30s	remaining: 12m 58s
100:	learn: 0.7544824	test: 0.1761477	best: 0.1775367 (99)	total: 3m 8s	remaining: 12m 25s
120:	learn: 0.7829476	test: 0.1875069	best: 0.1875069 (120)	total: 3m 46s	remaining: 11m 49s
140:	learn: 0.8030945	test: 0.2007947	best: 0.2020411 (138)	total: 4m 25s	remaining: 11m 15s
160:	learn: 0.8216801	test: 0.2101924	best: 0.2101924 (160)	total: 5m 2s	remaining: 10m 35s
180:	learn: 0.8668638	test: 0.2405553	best: 0.2405553 (180)	total: 5m 34s	remaining: 9m 49s
200:	learn: 0.9036235	test: 0.2627894	best: 0.2627894 (200)	total: 6m 6s	remaining: 9m 5s


## GradientBoost

In [ ]:
# 👉 Instala dependencias si aún no las tienes:
# !pip install pandas numpy scikit-learn imbalanced-learn xgboost catboost -q

import os
import time
from collections import Counter

# Hardcodear uso de 8 núcleos
os.environ["OMP_NUM_THREADS"]     = "8"
os.environ["OPENBLAS_NUM_THREADS"]= "8"
os.environ["MKL_NUM_THREADS"]     = "8"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

# ──────────── Step 1: Carga y preparación ────────────
t0 = time.time()
print("Step 1: Cargando datos...")
df = pd.read_csv("todoscsvs_transformado.csv")
print(f"  → {df.shape[0]}×{df.shape[1]} cargado in {time.time()-t0:.1f}s")

# Crear etiqueta y agrupar raros
measures = list("ABCDEFGHIJKLMNÑO")
df["target"] = df[measures].idxmax(axis=1)
df.loc[df[measures].sum(axis=1)==0, "target"] = "NoMeasure"
counts = df["target"].value_counts()
rare = counts[counts<100].index
df["target_grouped"] = df["target"].replace(rare, "Other")

X_raw = df.drop(columns=measures+["target","target_grouped"]).fillna("Missing")
y = df["target_grouped"]
le = LabelEncoder()
y_enc = le.fit_transform(y)
print(f"  → clases: {list(le.classes_)}")

# ──────────── Step 2: Split y one-hot vs raw ────────────
t0 = time.time()
X_oh = pd.get_dummies(X_raw, drop_first=True)
X_cb = X_raw.copy()
X_train_oh, X_test_oh, X_train_cb, X_test_cb, y_train, y_test = train_test_split(
    X_oh, X_cb, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)
print(f"Step 2: split in {time.time()-t0:.1f}s → train_oh {X_train_oh.shape}, test_oh {X_test_oh.shape}")

# ──────────── Step 3: Escalado (HGB/XGB) ────────────
t0 = time.time()
scaler    = StandardScaler(with_mean=False)
X_train_s = scaler.fit_transform(X_train_oh)
X_test_s  = scaler.transform(X_test_oh)
print(f"Step 3: escalado in {time.time()-t0:.1f}s")

# ──────────── Step 4: Balance moderado ────────────
t0 = time.time()
print("Step 4: Balance moderado basado en media de frecuencias...")
orig_counts = Counter(y_train)
mean_n      = int(np.mean(list(orig_counts.values())))
lower, upper = int(mean_n*0.5), int(mean_n*1.5)
print(f"  → original counts: {orig_counts}")
print(f"  → balancing range: [{lower}, {upper}]")

# Sobremuestreo para clases < lower
ros = RandomOverSampler(
    sampling_strategy={cls: lower for cls, cnt in orig_counts.items() if cnt < lower},
    random_state=42
)
X_ov, y_ov = ros.fit_resample(X_train_s, y_train)
print("  → after oversample:", Counter(y_ov))

# Submuestreo para clases > upper
ov_counts = Counter(y_ov)
rus = RandomUnderSampler(
    sampling_strategy={cls: upper for cls, cnt in ov_counts.items() if cnt > upper},
    random_state=42
)
X_bal, y_bal = rus.fit_resample(X_ov, y_ov)
print(f"  → final balanced: {Counter(y_bal)} ({time.time()-t0:.1f}s)")

# ──────────── Step 5: Sample weights ────────────
t0 = time.time()
bal_counts   = Counter(y_bal)
total_bal    = sum(bal_counts.values())
sample_w_bal = np.array([ total_bal / bal_counts[c] for c in y_bal ])
print(f"Step 5: sample weights ready in {time.time()-t0:.1f}s")

# ────────────────────────────────────
# A) HistGradientBoostingClassifier con early stopping
# ────────────────────────────────────
hgb_confs = [
    {"max_iter":100, "learning_rate":0.05, "max_depth":6},
    {"max_iter":80,  "learning_rate":0.1,  "max_depth":4},
    {"max_iter":120, "learning_rate":0.03, "max_depth":8},
]
print("\n=== HGB + early stopping ===")
for i, conf in enumerate(hgb_confs,1):
    print(f"\nStep 6.A.{i}: config={conf} → start training HGB")
    t1 = time.time()
    clf = HistGradientBoostingClassifier(
        loss="log_loss",
        validation_fraction=0.1,
        n_iter_no_change=10,
        early_stopping=True,
        verbose=1,
        random_state=42,
        **conf
    )
    clf.fit(X_bal, y_bal, sample_weight=sample_w_bal)
    print(f"  → HGB trained in {time.time()-t1:.1f}s")
    preds = clf.predict(X_test_s)
    print(classification_report(y_test, preds, zero_division=0))

# ────────────────────────────────────
# B) XGBoost multiclass con rounds reducidos
# ────────────────────────────────────
print("\n=== XGB + early stopping ===")
dtrain = xgb.DMatrix(X_bal, label=y_bal, weight=sample_w_bal)
dtest  = xgb.DMatrix(X_test_s, label=y_test)
xgb_confs = [
    {"eta":0.05, "max_depth":6, "subsample":0.8, "colsample_bytree":0.8},
    {"eta":0.1,  "max_depth":4, "subsample":0.7, "colsample_bytree":0.7},
    {"eta":0.03, "max_depth":8, "subsample":0.9, "colsample_bytree":0.9},
]
for i, conf in enumerate(xgb_confs,1):
    print(f"\nStep 6.B.{i}: config={conf} → start training XGB")
    t2 = time.time()
    params = {
        "objective":"multi:softmax",
        "num_class":len(le.classes_),
        "eval_metric":"mlogloss",
        "tree_method":"hist",
        "nthread":8,
        **conf
    }
    bst = xgb.train(
        params,
        dtrain,
        num_boost_round=100,
        early_stopping_rounds=10,
        evals=[(dtrain,"train"),(dtest,"eval")],
        verbose_eval=False
    )
    print(f"  → XGB trained in {time.time()-t2:.1f}s")
    preds = bst.predict(dtest).astype(int)
    print(classification_report(y_test, preds, zero_division=0))

# ────────────────────────────────────
# C) CatBoost con early stopping
# ────────────────────────────────────
print("\n=== CatBoost + early stopping ===")
cat_feats = X_train_cb.select_dtypes(include=["object","category"]).columns.tolist()
cw = {cls: len(y_train)/cnt for cls,cnt in Counter(y_train).items()}
pool_tr = Pool(X_train_cb, y_train, cat_features=cat_feats, weight=[cw[c] for c in y_train])
pool_ev = Pool(X_test_cb,  y_test,  cat_features=cat_feats)
cat_configs = [
    (200,0.05,6),
    (150,0.1,4),
    (250,0.03,8),
]
for i, (iters, lr, depth) in enumerate(cat_configs,1):
    print(f"\nStep 6.C.{i}: iters={iters}, lr={lr}, depth={depth} → start training CatBoost")
    t3 = time.time()
    model = CatBoostClassifier(
        iterations=iters,
        learning_rate=lr,
        depth=depth,
        loss_function="MultiClass",
        eval_metric="TotalF1",
        early_stopping_rounds=10,
        verbose=10,
        thread_count=8,
        class_weights=cw,
        random_seed=42
    )
    model.fit(pool_tr, eval_set=pool_ev)
    print(f"  → CatBoost trained in {time.time()-t3:.1f}s")
    preds = model.predict(X_test_cb)
    print(classification_report(y_test, preds, zero_division=0))


Step 1: Cargando datos...
  → 328959×85 cargado in 1.4s
  → clases: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'L', 'NoMeasure', 'Other']
Step 2: split in 4.5s → train_oh (263167, 2777), test_oh (65792, 2777)
Step 3: escalado in 28.8s
Step 4: Balance moderado basado en media de frecuencias...
  → Frecuencias originales: Counter({np.int64(11): 129755, np.int64(8): 97910, np.int64(0): 23682, np.int64(2): 5867, np.int64(1): 2711, np.int64(5): 1342, np.int64(6): 613, np.int64(9): 546, np.int64(4): 198, np.int64(7): 170, np.int64(3): 151, np.int64(12): 139, np.int64(10): 83})
  → Rango de balance: [10121, 30364]
  → Tras oversample: Counter({np.int64(11): 129755, np.int64(8): 97910, np.int64(0): 23682, np.int64(2): 10121, np.int64(1): 10121, np.int64(6): 10121, np.int64(5): 10121, np.int64(4): 10121, np.int64(10): 10121, np.int64(7): 10121, np.int64(9): 10121, np.int64(12): 10121, np.int64(3): 10121})
  → Distribución final: Counter({np.int64(8): 30364, np.int64(11): 30364, np.int64

NameError: name 'sample_w_bal' is not defined